
# AI / Private Credit / Insurance Stress Dashboard

A monitoring notebook for the thesis:

> **AI / data-center economics weaken → structured/private credit deteriorates → insurer capital pressure rises → funding stress appears → policy response follows.**

The dashboard deliberately separates **observable evidence** from the broader thesis. A deterioration in one layer does **not** imply insurer insolvency.

## Monitoring layers

1. **Compute economics**
2. **Credit markets**
3. **Private credit**
4. **Insurance balance sheets**
5. **Insurance funding / liquidity**
6. **Policy response**

### Public sources used automatically

- Federal Reserve / FRED
- Federal Reserve Financial Accounts (Z.1), via FRED

### File-driven sources

These do not have a dependable free real-time API, so the notebook accepts CSV inputs:

- Data-center ABS / structured-credit rating actions
- BDC PIK income, non-accruals, NAV changes
- NAIC statutory insurer metrics / RBC
- Bermuda reinsurance / BSCR metrics

The notebook will create empty templates for those files if they do not exist.


In [ ]:

# Optional installs:
# %pip install pandas numpy matplotlib requests

from pathlib import Path
from io import StringIO
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_DIR = Path("stress_dashboard_data")
DATA_DIR.mkdir(exist_ok=True)

START_DATE = "2018-01-01"
print(f"Data directory: {DATA_DIR.resolve()}")



## 1. Configuration

FRED's public graph CSV endpoint does not require an API key. The core automatic series are:

| Series | Meaning | Layer |
|---|---|---|
| `DGS10` | 10-year Treasury yield | Credit baseline |
| `BAMLC0A0CM` | ICE BofA US Corporate OAS | Credit |
| `BAMLH0A0HYM2` | ICE BofA US High Yield OAS | Credit |
| `WALCL` | Federal Reserve total assets | Policy response |
| `WPC` | Primary credit, weekly average | Policy / liquidity |
| `BOGZ1FL543190543Q` | Life insurers' funding agreements backing securities | Insurance funding |

You can add more FRED series to `FRED_SERIES`.


In [ ]:

FRED_SERIES = {
    "DGS10": "10Y Treasury Yield",
    "BAMLC0A0CM": "IG Corporate OAS",
    "BAMLH0A0HYM2": "HY Corporate OAS",
    "WALCL": "Fed Total Assets",
    "WPC": "Fed Primary Credit",
    "BOGZ1FL543190543Q": "Life Insurer FABS Liabilities",
}

def fetch_fred_csv(series_id: str, start: str = START_DATE) -> pd.Series:
    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={series_id}&cosd={start}"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df.set_index("date")[series_id]

def load_fred_panel(series_map=FRED_SERIES, start=START_DATE):
    series = {}
    errors = {}
    for sid, label in series_map.items():
        try:
            series[label] = fetch_fred_csv(sid, start)
        except Exception as e:
            errors[sid] = str(e)

    if not series:
        raise RuntimeError(f"No FRED series could be loaded. Errors: {errors}")

    panel = pd.concat(series, axis=1).sort_index()
    return panel, errors

fred, fred_errors = load_fred_panel()
print("Latest FRED observations:")
display(fred.ffill().tail(5))

if fred_errors:
    print("Series that failed:")
    display(pd.Series(fred_errors, name="error"))



## 2. Derived market indicators

The goal is not a black-box score. These are transparent transformations:

- **IG / HY OAS z-scores:** current spread relative to trailing 3-year history.
- **FABS growth:** year-over-year growth in life-insurer funding-agreement liabilities.
- **Primary-credit stress:** current borrowing relative to its own trailing distribution.
- **Fed balance-sheet impulse:** 13-week percentage change.

Higher values generally indicate more stress, except Fed balance-sheet growth, which is interpreted as a **policy-response** indicator rather than fundamental deterioration.


In [ ]:

def rolling_zscore(s: pd.Series, window=756, min_periods=126):
    mu = s.rolling(window, min_periods=min_periods).mean()
    sigma = s.rolling(window, min_periods=min_periods).std()
    return (s - mu) / sigma.replace(0, np.nan)

daily = fred.resample("D").ffill()

derived = pd.DataFrame(index=daily.index)

for col in ["IG Corporate OAS", "HY Corporate OAS", "Fed Primary Credit"]:
    if col in daily:
        derived[f"{col} z"] = rolling_zscore(daily[col])

if "Life Insurer FABS Liabilities" in daily:
    derived["FABS YoY %"] = daily["Life Insurer FABS Liabilities"].pct_change(365) * 100

if "Fed Total Assets" in daily:
    derived["Fed Assets 13w %"] = daily["Fed Total Assets"].pct_change(91) * 100

display(derived.tail(10))


In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(13, 11))

if "IG Corporate OAS" in daily:
    daily["IG Corporate OAS"].plot(ax=axes[0], label="IG OAS")
if "HY Corporate OAS" in daily:
    daily["HY Corporate OAS"].plot(ax=axes[0], label="HY OAS")
axes[0].set_title("Public Credit Spreads")
axes[0].set_ylabel("OAS")
axes[0].legend()

if "Life Insurer FABS Liabilities" in daily:
    daily["Life Insurer FABS Liabilities"].plot(ax=axes[1])
axes[1].set_title("Life Insurer Funding Agreements Backing Securities")
axes[1].set_ylabel("USD millions")

if "Fed Primary Credit" in daily:
    daily["Fed Primary Credit"].plot(ax=axes[2], label="Primary credit")
axes[2].set_title("Federal Reserve Primary Credit")
axes[2].set_ylabel("USD millions")

plt.tight_layout()
plt.show()



## 3. Structured-credit / data-center rating-action feed

Populate `rating_actions.csv` with rating actions from S&P, Moody's, Fitch, KBRA, Morningstar DBRS, etc.

Suggested focus:

- Data-center ABS / CMBS / securitizations
- Digital-infrastructure debt
- CLOs containing meaningful AI/data-center borrower exposure
- Senior tranches owned by insurers
- Negative outlook / watch changes even before a downgrade

The dashboard weights **senior-tranche downgrades** more heavily than junior-tranche actions because the thesis depends on deterioration reaching insurer-owned high-grade paper.


In [ ]:

rating_path = DATA_DIR / "rating_actions.csv"

rating_columns = [
    "date", "issuer", "deal", "asset_type", "agency",
    "tranche", "old_rating", "new_rating", "action",
    "seniority", "notional_usd_m", "ai_data_center_exposure_pct",
    "source_url", "notes",
]

if not rating_path.exists():
    pd.DataFrame(columns=rating_columns).to_csv(rating_path, index=False)

ratings = pd.read_csv(rating_path)
if len(ratings):
    ratings["date"] = pd.to_datetime(ratings["date"])
    ratings["notional_usd_m"] = pd.to_numeric(ratings["notional_usd_m"], errors="coerce")
    ratings["ai_data_center_exposure_pct"] = pd.to_numeric(
        ratings["ai_data_center_exposure_pct"], errors="coerce"
    )

display(ratings.tail(20))
print(f"Edit/import: {rating_path.resolve()}")


In [ ]:

ACTION_POINTS = {
    "upgrade": -1.0,
    "positive outlook": -0.5,
    "affirm": 0.0,
    "negative outlook": 1.0,
    "watch negative": 1.5,
    "downgrade": 2.0,
    "default": 4.0,
}

SENIORITY_MULTIPLIER = {
    "senior": 1.5,
    "mezzanine": 1.0,
    "junior": 0.7,
}

def rating_stress_score(df):
    if df.empty:
        return pd.DataFrame(columns=["date", "rating_stress"])
    x = df.copy()
    x["base"] = x["action"].str.lower().map(ACTION_POINTS).fillna(0)
    x["seniority_mult"] = x["seniority"].str.lower().map(SENIORITY_MULTIPLIER).fillna(1)
    exposure = x["ai_data_center_exposure_pct"].fillna(50).clip(0, 100) / 100
    notional = np.log1p(x["notional_usd_m"].fillna(100).clip(lower=0)) / np.log1p(1000)
    x["points"] = x["base"] * x["seniority_mult"] * (0.5 + exposure) * notional
    return (
        x.set_index("date")["points"]
        .resample("30D").sum()
        .rename("rating_stress")
        .to_frame()
    )

rating_stress = rating_stress_score(ratings)
display(rating_stress.tail(12))



## 4. Private-credit / BDC feed

For a first pass, track a basket of large BDCs / private-credit managers.

The strongest deterioration signals are usually:

- **Non-accruals % of portfolio**
- **PIK income / total investment income**
- **NAV change**
- **Interest coverage**
- **Realized credit losses**

PIK is useful because reported income can remain high while borrowers stop paying cash interest.


In [ ]:

bdc_path = DATA_DIR / "bdc_metrics.csv"

bdc_columns = [
    "date", "ticker", "manager",
    "non_accrual_pct", "pik_income_pct",
    "nav_per_share", "nav_qoq_pct",
    "realized_loss_pct", "interest_coverage",
    "source_url", "notes",
]

if not bdc_path.exists():
    pd.DataFrame(columns=bdc_columns).to_csv(bdc_path, index=False)

bdc = pd.read_csv(bdc_path)
if len(bdc):
    bdc["date"] = pd.to_datetime(bdc["date"])
    for c in [
        "non_accrual_pct", "pik_income_pct", "nav_per_share",
        "nav_qoq_pct", "realized_loss_pct", "interest_coverage"
    ]:
        bdc[c] = pd.to_numeric(bdc[c], errors="coerce")

display(bdc.tail(20))
print(f"Edit/import: {bdc_path.resolve()}")



## 5. Insurer statutory balance-sheet feed

Populate this from NAIC statutory filings / insurer statements.

The most useful normalized measures are against **capital & surplus**, because a $5B loss means very different things at insurers of different size.

Priority fields:

- RBC ratio
- Level 3 assets / capital & surplus
- Schedule BA / capital & surplus
- CLO + ABS / capital & surplus
- Unrealized losses / capital & surplus
- Reinsurance recoverables / capital & surplus
- Ceded reserves / total reserves
- Affiliated investments / capital & surplus
- NAIC-1 → lower designation migration


In [ ]:

insurer_path = DATA_DIR / "insurer_statutory.csv"

insurer_columns = [
    "date", "insurer", "parent",
    "total_assets_usd_m", "capital_surplus_usd_m", "rbc_ratio_pct",
    "level3_assets_usd_m", "schedule_ba_usd_m",
    "clo_abs_usd_m", "unrealized_losses_usd_m",
    "reinsurance_recoverables_usd_m", "ceded_reserves_usd_m",
    "total_reserves_usd_m", "affiliated_investments_usd_m",
    "naic1_to_lower_usd_m", "source_url", "notes",
]

if not insurer_path.exists():
    pd.DataFrame(columns=insurer_columns).to_csv(insurer_path, index=False)

insurers = pd.read_csv(insurer_path)
if len(insurers):
    insurers["date"] = pd.to_datetime(insurers["date"])
    numeric_cols = [c for c in insurer_columns if c.endswith("_usd_m") or c.endswith("_pct")]
    for c in numeric_cols:
        insurers[c] = pd.to_numeric(insurers[c], errors="coerce")

    cap = insurers["capital_surplus_usd_m"].replace(0, np.nan)
    insurers["level3_to_capital"] = insurers["level3_assets_usd_m"] / cap
    insurers["schedule_ba_to_capital"] = insurers["schedule_ba_usd_m"] / cap
    insurers["clo_abs_to_capital"] = insurers["clo_abs_usd_m"] / cap
    insurers["unrealized_loss_to_capital"] = insurers["unrealized_losses_usd_m"] / cap
    insurers["reinsurance_recoverables_to_capital"] = insurers["reinsurance_recoverables_usd_m"] / cap
    insurers["affiliated_investments_to_capital"] = insurers["affiliated_investments_usd_m"] / cap
    insurers["rating_migration_to_capital"] = insurers["naic1_to_lower_usd_m"] / cap

display(insurers.tail(20))
print(f"Edit/import: {insurer_path.resolve()}")



## 6. Bermuda / offshore reinsurance feed

Track the offshore leg separately rather than assuming the U.S. parent's balance sheet captures the risk.

Useful fields:

- Reinsurance assets
- Capital & surplus
- BSCR ratio
- Private credit / total assets
- Structured credit / total assets
- Related-party assets / capital
- U.S.-origin ceded reserves, where available


In [ ]:

bermuda_path = DATA_DIR / "bermuda_reinsurance.csv"

bermuda_columns = [
    "date", "entity",
    "reinsurance_assets_usd_m", "capital_surplus_usd_m",
    "bscr_ratio_pct", "private_credit_pct_assets",
    "structured_credit_pct_assets", "related_party_assets_usd_m",
    "us_origin_ceded_reserves_usd_m", "source_url", "notes",
]

if not bermuda_path.exists():
    pd.DataFrame(columns=bermuda_columns).to_csv(bermuda_path, index=False)

bermuda = pd.read_csv(bermuda_path)
if len(bermuda):
    bermuda["date"] = pd.to_datetime(bermuda["date"])
    for c in bermuda_columns:
        if c.endswith("_usd_m") or c.endswith("_pct"):
            bermuda[c] = pd.to_numeric(bermuda[c], errors="coerce")

display(bermuda.tail(20))
print(f"Edit/import: {bermuda_path.resolve()}")



## 7. Six-layer dashboard

This deliberately uses **status bands**, not a claim of probability of crisis.

- **0–1:** normal / low signal
- **1–2:** watch
- **2–3:** elevated
- **3+:** high stress

For public market variables the score uses rolling z-scores. File-driven components only contribute once you populate them.


In [ ]:

def latest_valid(s, default=np.nan):
    if s is None or len(s.dropna()) == 0:
        return default
    return float(s.dropna().iloc[-1])

def positive_z(s):
    z = latest_valid(s, 0.0)
    return max(0.0, z)

scores = {}

# 1. Compute economics: file-driven for now.
scores["1 Compute economics"] = np.nan

# 2. Credit: blend IG and HY OAS stress.
credit_parts = []
for c in ["IG Corporate OAS z", "HY Corporate OAS z"]:
    if c in derived:
        credit_parts.append(positive_z(derived[c]))
scores["2 Credit markets"] = np.mean(credit_parts) if credit_parts else np.nan

# 3. Private credit.
if len(bdc):
    latest_bdc = bdc.sort_values("date").groupby("ticker").tail(1)
    nonacc = latest_bdc["non_accrual_pct"].median()
    pik = latest_bdc["pik_income_pct"].median()
    nav = latest_bdc["nav_qoq_pct"].median()
    # Simple transparent heuristic. Tune after collecting history.
    scores["3 Private credit"] = (
        max(0, (nonacc - 1.0) / 2.0) +
        max(0, (pik - 5.0) / 10.0) +
        max(0, (-nav) / 5.0)
    )
else:
    scores["3 Private credit"] = np.nan

# 4. Insurance balance sheets.
if len(insurers):
    latest_ins = insurers.sort_values("date").groupby("insurer").tail(1)
    rbc = latest_ins["rbc_ratio_pct"].median()
    level3_cap = latest_ins["level3_to_capital"].median()
    unreal = latest_ins["unrealized_loss_to_capital"].median()
    migration = latest_ins["rating_migration_to_capital"].median()
    scores["4 Insurance balance sheets"] = (
        max(0, (300 - rbc) / 100) +
        max(0, level3_cap - 1.0) +
        max(0, unreal) +
        max(0, migration * 2)
    )
else:
    scores["4 Insurance balance sheets"] = np.nan

# 5. Funding/liquidity.
funding_parts = []
if "Fed Primary Credit z" in derived:
    funding_parts.append(positive_z(derived["Fed Primary Credit z"]))
if "FABS YoY %" in derived:
    fabs_growth = latest_valid(derived["FABS YoY %"], 0)
    # Rapid growth is treated as vulnerability, not proof of stress.
    funding_parts.append(max(0, (fabs_growth - 10) / 20))
scores["5 Funding / liquidity"] = np.mean(funding_parts) if funding_parts else np.nan

# 6. Policy response.
policy_parts = []
if "Fed Assets 13w %" in derived:
    impulse = latest_valid(derived["Fed Assets 13w %"], 0)
    policy_parts.append(max(0, impulse / 5))
if "Fed Primary Credit z" in derived:
    policy_parts.append(positive_z(derived["Fed Primary Credit z"]))
scores["6 Policy response"] = np.mean(policy_parts) if policy_parts else np.nan

score_df = pd.DataFrame(
    {"layer": list(scores.keys()), "stress_score": list(scores.values())}
)

def band(x):
    if pd.isna(x):
        return "NO DATA"
    if x < 1:
        return "LOW"
    if x < 2:
        return "WATCH"
    if x < 3:
        return "ELEVATED"
    return "HIGH"

score_df["status"] = score_df["stress_score"].map(band)
display(score_df)


In [ ]:

plot_df = score_df.dropna(subset=["stress_score"]).copy()

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(plot_df["layer"], plot_df["stress_score"])
ax.axvline(1, linestyle="--", linewidth=1)
ax.axvline(2, linestyle="--", linewidth=1)
ax.axvline(3, linestyle="--", linewidth=1)
ax.set_title("AI / Private Credit / Insurance Stress Dashboard")
ax.set_xlabel("Stress score (heuristic, not probability)")
plt.tight_layout()
plt.show()



## 8. Thesis confirmation matrix

The dashboard should be interpreted sequentially.

| Observation | Interpretation |
|---|---|
| Compute economics weakens, credit stays fine | AI economics issue; insurer thesis not confirmed |
| Credit spreads widen, but senior ratings stable | Market concern, limited regulatory-capital transmission |
| Senior data-center / structured debt downgraded | Major thesis milestone |
| BDC non-accrual + PIK rise | Private-credit deterioration broadening |
| Insurer RBC falls + rating migration rises | Balance-sheet transmission is occurring |
| FABS / insurer funding stress appears | Liquidity channel activating |
| Fed facilities / balance sheet expand | Policy-response phase |

The strongest evidence would be a **sequence**, not one red indicator.



## 9. Data-source notes

### FRED
The St. Louis Fed provides a REST API and public data downloads. The notebook uses the public graph CSV endpoint for convenience.

### Life-insurer FABS
Federal Reserve Financial Accounts series `BOGZ1FL543190543Q` measures life-insurance-company funding agreements backing securities, quarterly, end-of-period.

### NAIC
NAIC statutory filings provide Schedule D, Schedule BA, Schedule S, Schedule Y and RBC-related information. Beginning with year-end 2026 reporting, additional private-credit / private-rating / PIK-related disclosures should make this layer substantially more useful.

### Bermuda
The Bermuda Monetary Authority publishes long-term insurance market, reinsurance, liquidity, private-credit and stress-test reports. Those releases are currently best treated as periodic report ingestion rather than a high-frequency API.

## Next upgrades

1. Add automated **SEC/XBRL extraction** for BDC PIK, non-accruals and NAV.
2. Add a scraper/API adapter for **ratings-agency actions**.
3. Build an **NAIC statutory parser** for insurer-level Schedule D / BA / S data.
4. Add a **data-center deal registry** mapping borrower → facility → tenant → securitization → insurer holdings.
5. Add alert thresholds and historical backtesting.


In [ ]:

# Export a compact snapshot for use by cron/automation jobs.
snapshot = score_df.copy()
snapshot["as_of"] = pd.Timestamp.utcnow().isoformat()
snapshot_path = DATA_DIR / "latest_stress_snapshot.csv"
snapshot.to_csv(snapshot_path, index=False)

print(f"Saved snapshot: {snapshot_path.resolve()}")
display(snapshot)
